# Text2Cypher Retriever

Vector retrievers find data based on semantic similarity. But for specific questions about entities, relationships, or aggregations, you need Cypher queries.

The **Text2CypherRetriever** uses an LLM to convert natural language questions into Cypher queries.

**Examples of questions it handles well:**
- "What companies are owned by BlackRock?"
- "How many risk factors does Microsoft have?"
- "Which asset managers own both Apple and Amazon?"

---

## Setup

Import required modules and initialize connections.

In [ ]:
import sys
sys.path.insert(0, '..')

from neo4j import GraphDatabase
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.schema import get_schema

from config import get_neo4j_driver, get_llm

driver = get_neo4j_driver()
driver.verify_connectivity()

llm = get_llm()

print(f"Connected to Neo4j!")
print(f"LLM: {llm.model_id}")

## Get Database Schema

The Text2CypherRetriever needs the database schema to generate valid Cypher queries.

In [ ]:
schema = get_schema(driver)
print(schema)

## Custom Prompt for Modern Cypher

We provide a custom prompt that ensures the LLM generates modern, compatible Cypher syntax.

In [ ]:
TEXT2CYPHER_PROMPT = """Task: Generate a Cypher statement to query a graph database.

Instructions:
- Use only the provided relationship types and properties in the schema.
- Do not use any other relationship types or properties that are not provided.
- Only filter by name when a specific entity name is mentioned in the question.
  When filtering by name, use case-insensitive matching:
  `WHERE toLower(node.name) CONTAINS toLower('ActualEntityName')`
- Do NOT add name filters if no specific entity name is mentioned in the question.
- Always add LIMIT 20 to the end of your query to restrict results.

Modern Cypher Requirements:
- Use `elementId(node)` instead of `id(node)` (id() is removed in Neo4j 5+).
- Use `count{{pattern}}` instead of `size((pattern))` for counting patterns.
- Use `EXISTS {{MATCH pattern}}` instead of `exists((pattern))` for existence checks.
- When using ORDER BY, filter NULL values first: `WHERE property IS NOT NULL ORDER BY property`.
- Use explicit grouping with WITH clauses for aggregations.
- Limit collected results when appropriate: `collect(item)[0..20]`.

Schema:
{schema}

Note: Do not include any explanations or apologies in your responses.
Do not respond to any questions that might ask anything else than for you to construct a Cypher statement.
Do not include any text except the generated Cypher statement.

The question is:
{query_text}"""

print("Custom prompt defined!")

## Create Text2Cypher Retriever

Create the retriever with the schema and custom prompt.

In [ ]:
text2cypher_retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=schema,
    custom_prompt=TEXT2CYPHER_PROMPT
)

print("Text2CypherRetriever initialized!")

## Test Query Generation

Let's test the retriever with a specific entity query.

In [ ]:
query = "What companies are owned by BlackRock Inc."
cypher_result = text2cypher_retriever.get_search_results(query)

print(f"Original Query: {query}")
print(f"\nGenerated Cypher:\n{cypher_result.metadata['cypher']}")
print(f"\nNumber of results: {len(cypher_result.records)}")
print("\nResults:")
for record in cypher_result.records:
    print(f"  {record}")

**How it works:**
1. The natural language query is sent to Claude
2. Claude generates a Cypher query based on the schema
3. The Cypher query is executed against Neo4j
4. Results are returned with the generated Cypher for inspection

## GraphRAG with Text2Cypher

Use the retriever in a GraphRAG pipeline to generate natural language answers.

In [ ]:
query = "Who are the asset managers?"

rag = GraphRAG(llm=llm, retriever=text2cypher_retriever)
response = rag.search(query, return_context=True)

print(f"Query: \"{query}\"")
print(f"Number of results: {len(response.retriever_result.items)}\n")
print("Answer:")
print(response.answer)

In [ ]:
# View the generated Cypher
print("Generated Cypher:")
print(response.retriever_result.metadata["cypher"])

## Try Different Query Types

Text2Cypher excels at specific, fact-based questions.

In [ ]:
queries = [
    "How many risk factors does Apple face?",
    "What products does Microsoft mention?",
    "Which companies have filed documents?"
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"Query: \"{query}\"")
    print("="*60)
    
    response = rag.search(query, return_context=True)
    
    print(f"\nGenerated Cypher:\n{response.retriever_result.metadata['cypher']}")
    print(f"\nAnswer:\n{response.answer}")

## Complex Relationship Queries

Text2Cypher can handle complex relationship questions.

In [ ]:
query = "Which asset managers own both Apple and Microsoft?"

response = rag.search(query, return_context=True)

print(f"Query: \"{query}\"")
print(f"\nGenerated Cypher:\n{response.retriever_result.metadata['cypher']}")
print(f"\nAnswer:\n{response.answer}")

## Benefits of Text2Cypher

- **No manual Cypher writing** - The LLM generates queries from natural language
- **Accessible to non-technical users** - Anyone can query the graph
- **Rapid prototyping** - Quickly explore your data without writing Cypher
- **Fact-based answers** - Gets specific, structured data from the graph

> **Tip:** You can provide few-shot `examples` when creating the Text2CypherRetriever to improve query generation for your specific domain.

## Summary

In this notebook, you learned:

1. **Text2CypherRetriever** - Converts natural language to Cypher
2. **Schema-aware generation** - Uses your graph schema for valid queries
3. **Custom prompts** - Ensuring modern, compatible Cypher syntax
4. **Fact-based retrieval** - Getting specific answers from structured data

### When to Use Each Retriever

| Retriever | Best For |
|-----------|----------|
| **Vector** | Semantic similarity, exploratory questions |
| **VectorCypher** | Context + relationships, comparative insights |
| **Text2Cypher** | Specific facts, counts, entity queries |

---

**Next:** Continue to [Lab 8 - GraphRAG Agents](../Lab_8_Agents/) to build agents that can select and combine these retrieval patterns automatically.

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")